# Baseline Evaluation Notebook

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Setup & Install Dependencies

In [ ]:
%pip install transformers jsonlines tqdm

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import jsonlines
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report
from tqdm import tqdm

## Step 2: Load the Tokenizer and Model

In [ ]:
# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Load trained model
model_path = "/content/drive/MyDrive/SemEval-2024 Task 8/models/baseline/baseline_roberta.pth"
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model.load_state_dict(torch.load(model_path))
model.eval()  # Set model to evaluation mode

## Step 3: Define Text Dataset Class

In [ ]:
class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=512):
        self.texts = []
        self.labels = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Load JSONL file
        with jsonlines.open(file_path) as reader:
            for obj in reader:
                self.texts.append(obj["text"])
                self.labels.append(obj["label"])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

## Step 4: Load the Test Dataset

In [ ]:
# Define test dataset path
test_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_monolingual.jsonl"

# Load dataset
test_dataset = TextDataset(test_file, tokenizer)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## Step 5: Evaluate the Model

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds = []
all_labels = []

# Disable gradient computation during evaluation
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # Get model predictions

        # Convert logits to predicted labels
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = labels.cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

# Print classification report
print(classification_report(all_labels, all_preds, digits=4))

Evaluating: 100%|██████████| 2142/2142 [09:21<00:00,  3.82it/s]

              precision    recall  f1-score   support

           0     0.8332    0.6394    0.7236     16272
           1     0.7307    0.8843    0.8002     18000

    accuracy                         0.7680     34272
   macro avg     0.7819    0.7619    0.7619     34272
weighted avg     0.7794    0.7680    0.7638     34272

